# SQL Mastery for Quantitative Research

## Professional SQL foundations for systematic trading workflows

This notebook is a **reference-and-recap notebook**, not an exercise sheet. Its objective is to make the SQL constructs required for quantitative research familiar enough that they can later be used independently in a TODO-based case study.

The focus is analytical SQL: retrieving, joining, validating, transforming, ranking, and aggregating market data before handing a research-ready dataset to pandas, NumPy, scikit-learn, or PyTorch.

### Learning objectives

By the end of the notebook, you should be comfortable with:

- relational tables, rows, columns, primary keys, and foreign keys;
- `SELECT`, aliases, expressions, `DISTINCT`, filtering, sorting, and limiting;
- `NULL` semantics and `COALESCE`;
- conditional logic with `CASE`;
- aggregation with `GROUP BY` and `HAVING`;
- joins and the duplicate-row risks they introduce;
- subqueries and common table expressions (CTEs);
- date extraction and date arithmetic;
- window functions, including `LAG`, `LEAD`, ranking, cumulative and rolling calculations;
- reshaping with conditional aggregation;
- set operations;
- data-quality checks;
- query organization and the division of work between SQL and pandas.

The examples use **DuckDB SQL** because it is lightweight and well suited to analytical workflows. The SQL concepts are largely portable to PostgreSQL and other relational databases, although some date and utility functions differ by dialect.


## 0. Setup

The notebook creates a small synthetic market database with four tables:

- `assets`: security metadata;
- `prices`: daily close prices and volumes;
- `signals`: daily model signals;
- `trades`: example executions.

The data is deliberately small enough to inspect while preserving the table structure used in real research databases.


In [1]:
# Run once if DuckDB is not installed:
# %pip install duckdb

import duckdb
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
con = duckdb.connect()

dates = pd.bdate_range("2024-01-02", periods=90)
assets = pd.DataFrame({
    "asset": ["EQ_US", "EQ_EU", "BOND_US", "BOND_EU", "GOLD", "OIL", "EURUSD", "USDJPY"],
    "asset_class": ["Equity", "Equity", "Rates", "Rates", "Commodity", "Commodity", "FX", "FX"],
    "region": ["US", "Europe", "US", "Europe", "Global", "Global", "Europe", "Japan"],
    "currency": ["USD", "EUR", "USD", "EUR", "USD", "USD", "USD", "JPY"],
})

rows = []
signal_rows = []
for j, asset in enumerate(assets["asset"]):
    rets = rng.normal(0.0002 + j * 0.00001, 0.008 + j * 0.0003, len(dates))
    close = (100 + 5 * j) * np.exp(np.cumsum(rets))
    volume = rng.integers(100_000, 2_000_000, len(dates))
    signal = pd.Series(rets).rolling(10, min_periods=3).mean().to_numpy() + rng.normal(0, 0.002, len(dates))
    rows.extend(zip(dates, [asset] * len(dates), close, volume))
    signal_rows.extend(zip(dates, [asset] * len(dates), signal))

prices = pd.DataFrame(rows, columns=["date", "asset", "close", "volume"])
signals = pd.DataFrame(signal_rows, columns=["date", "asset", "signal"])

trade_dates = dates[::9]
trades = pd.DataFrame({
    "trade_id": np.arange(1, len(trade_dates) * 3 + 1),
    "date": np.repeat(trade_dates, 3),
    "asset": np.tile(["EQ_US", "GOLD", "EURUSD"], len(trade_dates)),
    "quantity": rng.integers(-250, 251, len(trade_dates) * 3),
    "price": rng.normal(100, 8, len(trade_dates) * 3),
})

for df in [assets, prices, signals, trades]:
    string_cols = df.select_dtypes(include=["str", "string"]).columns
    df[string_cols] = df[string_cols].astype("object")

con.register("assets_df", assets)
con.register("prices_df", prices)
con.register("signals_df", signals)
con.register("trades_df", trades)

con.execute("CREATE OR REPLACE TABLE assets AS SELECT * FROM assets_df")
con.execute("CREATE OR REPLACE TABLE prices AS SELECT * FROM prices_df")
con.execute("CREATE OR REPLACE TABLE signals AS SELECT * FROM signals_df")
con.execute("CREATE OR REPLACE TABLE trades AS SELECT * FROM trades_df")


### Helper

`con.sql(...)` returns a DuckDB relation. Calling `.df()` converts the result to a pandas DataFrame. In a production workflow, SQL should normally reduce and shape the data **before** that conversion.


In [2]:
con.sql("SELECT * FROM assets ORDER BY asset").df()


,asset,asset_class,region,currency
0,BOND_EU,Rates,Europe,EUR
1,BOND_US,Rates,US,USD
2,EQ_EU,Equity,Europe,EUR
3,EQ_US,Equity,US,USD
4,EURUSD,FX,Europe,USD
5,GOLD,Commodity,Global,USD
6,OIL,Commodity,Global,USD
7,USDJPY,FX,Japan,JPY


## 1. Relational structure and keys

A relational database stores data in tables connected by keys.

For this notebook:

- `assets.asset` identifies an asset in the metadata table;
- `(prices.date, prices.asset)` should identify one price observation;
- `(signals.date, signals.asset)` should identify one signal observation;
- `trades.trade_id` identifies one execution.

A **primary key** uniquely identifies a row. A **foreign key** links a value to another table. Even when a research warehouse does not enforce these constraints physically, you should reason about the data as if the intended keys were explicit.

Before writing a complex query, ask:

1. What does one row represent?
2. Which columns uniquely identify a row?
3. What cardinality should a join have?


In [3]:
con.sql('''
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT asset) AS n_assets,
    COUNT(DISTINCT date) AS n_dates
FROM prices
''').df()


,n_rows,n_assets,n_dates
0,720,8,90


## 2. SELECT, expressions, aliases, DISTINCT, ORDER BY, LIMIT

`SELECT` chooses columns and can compute new expressions. `AS` gives an output column a clear name.

SQL is declarative: you describe the result you want rather than iterating row by row.


In [4]:
con.sql('''
SELECT
    date,
    asset,
    close,
    volume,
    close * volume AS dollar_volume
FROM prices
ORDER BY date DESC, asset
LIMIT 10
''').df()


,date,asset,close,volume,dollar_volume
0,2024-05-06,BOND_EU,116.136092,365653,4.246551e+07
1,2024-05-06,BOND_US,88.365545,1909603,1.687431e+08
2,2024-05-06,EQ_EU,102.264988,1080706,1.105184e+08
3,2024-05-06,EQ_US,90.621577,944508,8.559280e+07
4,2024-05-06,EURUSD,123.232311,1535622,1.892382e+08
5,2024-05-06,GOLD,135.649134,1628495,2.209039e+08
6,2024-05-06,OIL,117.867431,634246,7.475695e+07
7,2024-05-06,USDJPY,128.937502,889855,1.147357e+08
8,2024-05-03,BOND_EU,116.356272,1499585,1.744861e+08
9,2024-05-03,BOND_US,89.372294,478303,4.274704e+07


`DISTINCT` removes duplicate combinations from the selected columns. Use it intentionally; it should not be used to hide a join that accidentally duplicated rows.


In [5]:
con.sql('''
SELECT DISTINCT asset
FROM prices
ORDER BY asset
''').df()


,asset
0,BOND_EU
1,BOND_US
2,EQ_EU
3,EQ_US
4,EURUSD
5,GOLD
6,OIL
7,USDJPY


## 3. Filtering: WHERE, IN, BETWEEN, LIKE

`WHERE` filters rows **before** aggregation.

Common predicates include:

- comparisons: `=`, `<>`, `>`, `>=`, `<`, `<=`;
- membership: `IN (...)`;
- ranges: `BETWEEN ... AND ...`;
- pattern matching: `LIKE`;
- boolean combinations: `AND`, `OR`, `NOT`.

Use parentheses when mixing `AND` and `OR`.


In [6]:
con.sql('''
SELECT date, asset, close
FROM prices
WHERE date BETWEEN DATE '2024-02-01' AND DATE '2024-02-29'
  AND asset IN ('EQ_US', 'EQ_EU')
ORDER BY date, asset
''').df()


,date,asset,close
0,2024-02-01,EQ_EU,101.954788
1,2024-02-01,EQ_US,92.980514
2,2024-02-02,EQ_EU,101.392109
3,2024-02-02,EQ_US,93.201149
4,2024-02-05,EQ_EU,99.769672
5,2024-02-05,EQ_US,93.336763
6,2024-02-06,EQ_EU,97.133596
7,2024-02-06,EQ_US,93.215929
8,2024-02-07,EQ_EU,96.727462
9,2024-02-07,EQ_US,91.376153


## 4. NULL: missing information is not zero

SQL uses `NULL` for missing or unknown values. Comparisons such as `value = NULL` do not work because `NULL` is not an ordinary value.

Use:

- `IS NULL`;
- `IS NOT NULL`;
- `COALESCE(x, fallback)`.

Aggregations such as `AVG`, `SUM`, and `STDDEV_SAMP` generally ignore `NULL` observations.


In [7]:
con.sql('''
SELECT
    date,
    asset,
    signal,
    COALESCE(signal, 0.0) AS signal_filled
FROM signals
WHERE signal IS NULL
ORDER BY date, asset
LIMIT 10
''').df()


,date,asset,signal,signal_filled
0,2024-01-02,BOND_EU,NaN,0.0
1,2024-01-02,BOND_US,NaN,0.0
2,2024-01-02,EQ_EU,NaN,0.0
3,2024-01-02,EQ_US,NaN,0.0
4,2024-01-02,EURUSD,NaN,0.0
5,2024-01-02,GOLD,NaN,0.0
6,2024-01-02,OIL,NaN,0.0
7,2024-01-02,USDJPY,NaN,0.0
8,2024-01-03,BOND_EU,NaN,0.0
9,2024-01-03,BOND_US,NaN,0.0


## 5. Conditional logic with CASE

`CASE` is SQL's general conditional expression. It is analogous to vectorized conditional logic such as `np.select` or chained boolean masks.

The result can be numeric, text, dates, or other compatible types.


In [8]:
con.sql('''
SELECT
    date,
    asset,
    signal,
    CASE
        WHEN signal > 0.002 THEN 'Positive'
        WHEN signal < -0.002 THEN 'Negative'
        ELSE 'Neutral'
    END AS signal_bucket
FROM signals
WHERE signal IS NOT NULL
ORDER BY date, asset
LIMIT 12
''').df()


,date,asset,signal,signal_bucket
0,2024-01-04,BOND_EU,0.001929,Neutral
1,2024-01-04,BOND_US,-0.001115,Neutral
2,2024-01-04,EQ_EU,-0.003391,Negative
3,2024-01-04,EQ_US,-0.000279,Neutral
4,2024-01-04,EURUSD,-0.007267,Negative
5,2024-01-04,GOLD,-0.002066,Negative
6,2024-01-04,OIL,0.001444,Neutral
7,2024-01-04,USDJPY,-0.000917,Neutral
8,2024-01-05,BOND_EU,-0.003163,Negative
9,2024-01-05,BOND_US,-0.001443,Neutral


## 6. Aggregation: GROUP BY

Aggregate functions collapse multiple rows into summary values.

Core functions:

- `COUNT(*)`;
- `COUNT(column)`;
- `COUNT(DISTINCT column)`;
- `SUM`;
- `AVG`;
- `MIN`, `MAX`;
- `STDDEV_SAMP`.

Every selected column that is not aggregated must normally appear in `GROUP BY`.


In [9]:
con.sql('''
SELECT
    asset,
    AVG(close) AS avg_close,
    STDDEV_SAMP(close) AS close_std,
    AVG(volume) AS avg_volume,
    COUNT(*) AS n_obs
FROM prices
GROUP BY asset
ORDER BY asset
''').df()


,asset,avg_close,close_std,avg_volume,n_obs
0,BOND_EU,114.626282,3.194022,1.128481e+06,90
1,BOND_US,98.563092,6.482462,1.000057e+06,90
2,EQ_EU,100.315731,2.530272,1.020901e+06,90
3,EQ_US,92.121923,3.795014,1.101443e+06,90
4,EURUSD,119.503939,3.388845,1.008067e+06,90
5,GOLD,130.491818,6.106603,1.046964e+06,90
6,OIL,121.294817,3.986391,1.069986e+06,90
7,USDJPY,130.816557,5.183989,1.025033e+06,90


## 7. WHERE versus HAVING

The distinction is fundamental:

- `WHERE` filters **rows before grouping**;
- `HAVING` filters **groups after aggregation**.

If the condition depends on an aggregate such as `AVG(...)` or `COUNT(...)`, it belongs in `HAVING`.


In [10]:
con.sql('''
SELECT
    asset,
    AVG(volume) AS avg_volume
FROM prices
WHERE date >= DATE '2024-02-01'
GROUP BY asset
HAVING AVG(volume) > 900000
ORDER BY avg_volume DESC
''').df()


,asset,avg_volume
0,OIL,1.130125e+06
1,BOND_EU,1.128817e+06
2,EQ_US,1.107646e+06
3,GOLD,1.081210e+06
4,BOND_US,1.065114e+06
5,EQ_EU,1.033239e+06
6,USDJPY,9.972718e+05
7,EURUSD,9.914852e+05


## 8. Joins

Joins combine tables through matching keys.

The main forms you need are:

- `INNER JOIN`: keep matched rows only;
- `LEFT JOIN`: keep every row from the left table and attach matches from the right;
- `FULL OUTER JOIN`: retain unmatched rows from both sides.

For research work, the most important issue is often not syntax but **join cardinality**.


In [11]:
con.sql('''
SELECT
    p.date,
    p.asset,
    p.close,
    a.asset_class,
    a.region
FROM prices AS p
INNER JOIN assets AS a
    ON p.asset = a.asset
ORDER BY p.date, p.asset
LIMIT 12
''').df()


,date,asset,close,asset_class,region
0,2024-01-02,BOND_EU,115.146165,Rates,Europe
1,2024-01-02,BOND_US,109.809356,Rates,US
2,2024-01-02,EQ_EU,105.377015,Equity,Europe
3,2024-01-02,EQ_US,100.020986,Equity,US
4,2024-01-02,EURUSD,129.067028,FX,Europe
5,2024-01-02,GOLD,118.965695,Commodity,Global
6,2024-01-02,OIL,125.101562,Commodity,Global
7,2024-01-02,USDJPY,135.037698,FX,Japan
8,2024-01-03,BOND_EU,115.636404,Rates,Europe
9,2024-01-03,BOND_US,109.123596,Rates,US


### Join cardinality and accidental duplication

Suppose `(date, asset)` is unique in `prices` and `signals`. Joining on both columns should preserve one row per price observation.

Joining only on `asset` would create many-to-many matches across dates and explode the row count.

A useful professional habit is to check row counts and key uniqueness before and after important joins.


In [12]:
con.sql('''
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT (date, asset)) AS n_unique_date_asset
FROM (
    SELECT p.date, p.asset, p.close, s.signal
    FROM prices AS p
    LEFT JOIN signals AS s
      ON p.date = s.date
     AND p.asset = s.asset
)
''').df()


,n_rows,n_unique_date_asset
0,720,720


## 9. Subqueries

A subquery is a query nested inside another query. It is useful when an intermediate result is needed only once.

The example below selects observations whose volume is above that asset's full-sample average.


In [13]:
con.sql('''
SELECT p.date, p.asset, p.volume
FROM prices AS p
WHERE p.volume > (
    SELECT AVG(p2.volume)
    FROM prices AS p2
    WHERE p2.asset = p.asset
)
ORDER BY p.asset, p.date
LIMIT 15
''').df()


,date,asset,volume
0,2024-01-02,BOND_EU,1945505
1,2024-01-05,BOND_EU,1244697
2,2024-01-08,BOND_EU,1609833
3,2024-01-09,BOND_EU,1877622
4,2024-01-10,BOND_EU,1395383
5,2024-01-15,BOND_EU,1453489
6,2024-01-16,BOND_EU,1664867
7,2024-01-18,BOND_EU,1718307
8,2024-01-23,BOND_EU,1948050
9,2024-01-25,BOND_EU,1421961


## 10. Common table expressions (WITH)

CTEs give names to intermediate query steps. For analytical work, they are often clearer than deeply nested subqueries.

Think of each CTE as a well-defined transformation stage. This is close to assigning intermediate DataFrames in pandas, but the database can optimize the complete query plan.


In [14]:
con.sql('''
WITH price_returns AS (
    SELECT
        date,
        asset,
        close / LAG(close) OVER (
            PARTITION BY asset
            ORDER BY date
        ) - 1 AS ret
    FROM prices
),
asset_stats AS (
    SELECT
        asset,
        AVG(ret) AS mean_ret,
        STDDEV_SAMP(ret) AS vol
    FROM price_returns
    GROUP BY asset
)
SELECT *
FROM asset_stats
ORDER BY vol DESC
''').df()


,asset,mean_ret,vol
0,EURUSD,-0.000451,0.011775
1,USDJPY,-0.000477,0.009239
2,BOND_EU,0.000138,0.009217
3,OIL,-0.000627,0.009201
4,GOLD,0.001514,0.008852
5,EQ_EU,-0.000305,0.008038
6,BOND_US,-0.002407,0.007942
7,EQ_US,-0.001084,0.006942


## 11. Dates

Dates are first-class SQL values. Useful operations include:

- typed literals such as `DATE '2024-01-01'`;
- `EXTRACT(...)`;
- `DATE_TRUNC(...)`;
- interval arithmetic.

Exact date syntax varies more across SQL dialects than basic `SELECT`/`JOIN` syntax.


In [15]:
con.sql('''
SELECT
    date,
    EXTRACT(YEAR FROM date) AS year,
    EXTRACT(MONTH FROM date) AS month,
    DATE_TRUNC('month', date) AS month_start
FROM prices
GROUP BY date
ORDER BY date
LIMIT 10
''').df()


,date,year,month,month_start
0,2024-01-02,2024,1,2024-01-01
1,2024-01-03,2024,1,2024-01-01
2,2024-01-04,2024,1,2024-01-01
3,2024-01-05,2024,1,2024-01-01
4,2024-01-08,2024,1,2024-01-01
5,2024-01-09,2024,1,2024-01-01
6,2024-01-10,2024,1,2024-01-01
7,2024-01-11,2024,1,2024-01-01
8,2024-01-12,2024,1,2024-01-01
9,2024-01-15,2024,1,2024-01-01


## 12. Window functions: the analytical SQL core

Window functions calculate across related rows **without collapsing them**.

General form:

```sql
FUNCTION(...) OVER (
    PARTITION BY ...
    ORDER BY ...
    ROWS BETWEEN ...
)
```

`PARTITION BY` defines independent groups. `ORDER BY` defines sequence within each group.

This family is especially important for market data.


### 12.1 LAG and LEAD

`LAG(x)` accesses a previous row within the ordered partition. `LEAD(x)` accesses a subsequent row.

For prices, `LAG` is the natural SQL analogue of a grouped `shift(1)`.


In [16]:
con.sql('''
SELECT
    date,
    asset,
    close,
    LAG(close) OVER (
        PARTITION BY asset
        ORDER BY date
    ) AS previous_close,
    close / LAG(close) OVER (
        PARTITION BY asset
        ORDER BY date
    ) - 1 AS simple_return
FROM prices
ORDER BY asset, date
LIMIT 15
''').df()


,date,asset,close,previous_close,simple_return
0,2024-01-02,BOND_EU,115.146165,NaN,NaN
1,2024-01-03,BOND_EU,115.636404,115.146165,0.004258
2,2024-01-04,BOND_EU,114.812618,115.636404,-0.007124
3,2024-01-05,BOND_EU,113.168735,114.812618,-0.014318
4,2024-01-08,BOND_EU,111.756572,113.168735,-0.012478
5,2024-01-09,BOND_EU,112.446253,111.756572,0.006171
6,2024-01-10,BOND_EU,111.715539,112.446253,-0.006498
7,2024-01-11,BOND_EU,111.600755,111.715539,-0.001027
8,2024-01-12,BOND_EU,111.837798,111.600755,0.002124
9,2024-01-15,BOND_EU,112.481633,111.837798,0.005757


### 12.2 Cross-sectional ranking

Ranking functions include:

- `ROW_NUMBER`: unique sequential positions;
- `RANK`: ties share a rank and leave gaps;
- `DENSE_RANK`: ties share a rank without gaps;
- `NTILE(n)`: approximately splits ordered observations into `n` buckets.

For cross-sectional signals, the partition is often the date.


In [17]:
con.sql('''
SELECT
    date,
    asset,
    signal,
    RANK() OVER (
        PARTITION BY date
        ORDER BY signal
    ) AS signal_rank,
    NTILE(4) OVER (
        PARTITION BY date
        ORDER BY signal
    ) AS signal_quartile
FROM signals
WHERE signal IS NOT NULL
ORDER BY date, signal_rank
LIMIT 20
''').df()


,date,asset,signal,signal_rank,signal_quartile
0,2024-01-04,EURUSD,-0.007267,1,1
1,2024-01-04,EQ_EU,-0.003391,2,1
2,2024-01-04,GOLD,-0.002066,3,2
3,2024-01-04,BOND_US,-0.001115,4,2
4,2024-01-04,USDJPY,-0.000917,5,3
5,2024-01-04,EQ_US,-0.000279,6,3
6,2024-01-04,OIL,0.001444,7,4
7,2024-01-04,BOND_EU,0.001929,8,4
8,2024-01-05,EURUSD,-0.007329,1,1
9,2024-01-05,GOLD,-0.003590,2,1


### 12.3 Cumulative calculations

A windowed aggregate can preserve every row while calculating a cumulative statistic.

Explicitly writing the frame is good practice because default window frames can differ in ways that matter.


In [18]:
con.sql('''
WITH returns AS (
    SELECT
        date,
        asset,
        close / LAG(close) OVER (
            PARTITION BY asset
            ORDER BY date
        ) - 1 AS ret
    FROM prices
)
SELECT
    date,
    asset,
    ret,
    SUM(ret) OVER (
        PARTITION BY asset
        ORDER BY date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_arithmetic_return
FROM returns
ORDER BY asset, date
LIMIT 15
''').df()


,date,asset,ret,cumulative_arithmetic_return
0,2024-01-02,BOND_EU,NaN,NaN
1,2024-01-03,BOND_EU,0.004258,0.004258
2,2024-01-04,BOND_EU,-0.007124,-0.002866
3,2024-01-05,BOND_EU,-0.014318,-0.017184
4,2024-01-08,BOND_EU,-0.012478,-0.029663
5,2024-01-09,BOND_EU,0.006171,-0.023491
6,2024-01-10,BOND_EU,-0.006498,-0.029990
7,2024-01-11,BOND_EU,-0.001027,-0.031017
8,2024-01-12,BOND_EU,0.002124,-0.028893
9,2024-01-15,BOND_EU,0.005757,-0.023136


### 12.4 Rolling windows

A fixed trailing window uses a bounded frame. The example below computes a 20-observation rolling sample volatility.

`ROWS BETWEEN 19 PRECEDING AND CURRENT ROW` means at most 20 rows including the current row.


In [19]:
con.sql('''
WITH returns AS (
    SELECT
        date,
        asset,
        close / LAG(close) OVER (
            PARTITION BY asset
            ORDER BY date
        ) - 1 AS ret
    FROM prices
)
SELECT
    date,
    asset,
    ret,
    STDDEV_SAMP(ret) OVER (
        PARTITION BY asset
        ORDER BY date
        ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
    ) AS rolling_vol_20
FROM returns
ORDER BY asset, date
LIMIT 30
''').df()


,date,asset,ret,rolling_vol_20
0,2024-01-02,BOND_EU,NaN,NaN
1,2024-01-03,BOND_EU,0.004258,NaN
2,2024-01-04,BOND_EU,-0.007124,0.008048
3,2024-01-05,BOND_EU,-0.014318,0.009366
4,2024-01-08,BOND_EU,-0.012478,0.008359
5,2024-01-09,BOND_EU,0.006171,0.009451
6,2024-01-10,BOND_EU,-0.006498,0.008485
7,2024-01-11,BOND_EU,-0.001027,0.007890
8,2024-01-12,BOND_EU,0.002124,0.007664
9,2024-01-15,BOND_EU,0.005757,0.007819


### 12.5 Expanding calculations

An expanding window starts at the beginning of the partition:

```sql
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
```

If today's classification must exclude today's observation, end the frame at `1 PRECEDING`.


In [20]:
con.sql('''
WITH returns AS (
    SELECT
        date,
        asset,
        close / LAG(close) OVER (
            PARTITION BY asset
            ORDER BY date
        ) - 1 AS ret
    FROM prices
)
SELECT
    date,
    asset,
    ret,
    AVG(ret) OVER (
        PARTITION BY asset
        ORDER BY date
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS historical_mean_ex_today
FROM returns
ORDER BY asset, date
LIMIT 20
''').df()


,date,asset,ret,historical_mean_ex_today
0,2024-01-02,BOND_EU,NaN,NaN
1,2024-01-03,BOND_EU,0.004258,NaN
2,2024-01-04,BOND_EU,-0.007124,0.004258
3,2024-01-05,BOND_EU,-0.014318,-0.001433
4,2024-01-08,BOND_EU,-0.012478,-0.005728
5,2024-01-09,BOND_EU,0.006171,-0.007416
6,2024-01-10,BOND_EU,-0.006498,-0.004698
7,2024-01-11,BOND_EU,-0.001027,-0.004998
8,2024-01-12,BOND_EU,0.002124,-0.004431
9,2024-01-15,BOND_EU,0.005757,-0.003612


## 13. QUALIFY: filtering after window functions

DuckDB supports `QUALIFY`, which filters rows after window functions have been evaluated.

This is useful for tasks such as "top 2 signals per date" without adding another CTE. PostgreSQL does not currently use `QUALIFY` in the same way, so a subquery/CTE is the more portable pattern.


In [21]:
con.sql('''
SELECT
    date,
    asset,
    signal,
    RANK() OVER (
        PARTITION BY date
        ORDER BY signal DESC
    ) AS rank_desc
FROM signals
WHERE signal IS NOT NULL
QUALIFY rank_desc <= 2
ORDER BY date, rank_desc
LIMIT 20
''').df()


,date,asset,signal,rank_desc
0,2024-01-04,BOND_EU,0.001929,1
1,2024-01-04,OIL,0.001444,2
2,2024-01-05,OIL,0.000190,1
3,2024-01-05,BOND_US,-0.001443,2
4,2024-01-08,USDJPY,0.003629,1
5,2024-01-08,EQ_EU,-0.001238,2
6,2024-01-09,OIL,0.005319,1
7,2024-01-09,USDJPY,0.004581,2
8,2024-01-10,USDJPY,0.001202,1
9,2024-01-10,OIL,0.000261,2


## 14. Conditional aggregation and pivot-style output

SQL often reshapes categories into columns through conditional aggregation.

This is particularly useful for compact reporting tables.


In [22]:
con.sql('''
SELECT
    date,
    AVG(CASE WHEN a.asset_class = 'Equity' THEN p.close END) AS equity_avg_close,
    AVG(CASE WHEN a.asset_class = 'Rates' THEN p.close END) AS rates_avg_close,
    AVG(CASE WHEN a.asset_class = 'Commodity' THEN p.close END) AS commodity_avg_close,
    AVG(CASE WHEN a.asset_class = 'FX' THEN p.close END) AS fx_avg_close
FROM prices AS p
JOIN assets AS a
  ON p.asset = a.asset
GROUP BY date
ORDER BY date
LIMIT 10
''').df()


,date,equity_avg_close,rates_avg_close,commodity_avg_close,fx_avg_close
0,2024-01-02,102.699001,112.477760,122.033629,132.052363
1,2024-01-03,102.408807,112.380000,121.856089,131.311074
2,2024-01-04,102.035183,112.256626,122.067728,131.305383
3,2024-01-05,101.271332,111.373678,121.963545,130.493251
4,2024-01-08,100.733395,110.964679,121.449379,131.080975
5,2024-01-09,100.445785,111.299235,121.700612,130.349097
6,2024-01-10,100.157655,110.433976,122.367218,129.286985
7,2024-01-11,100.859320,110.340696,123.227328,127.878943
8,2024-01-12,100.829040,110.495598,124.106482,127.630309
9,2024-01-15,101.474149,111.281345,123.612843,127.169658


## 15. Set operations

Set operations combine compatible query results:

- `UNION` combines and removes duplicates;
- `UNION ALL` combines and preserves duplicates;
- `INTERSECT` keeps common rows;
- `EXCEPT` keeps rows in the first result but not the second.

`UNION ALL` is usually preferable when duplicate removal is not required because it avoids unnecessary work.


In [23]:
con.sql('''
SELECT asset FROM assets WHERE asset_class = 'Equity'
UNION ALL
SELECT asset FROM assets WHERE asset_class = 'Commodity'
ORDER BY asset
''').df()


,asset
0,EQ_EU
1,EQ_US
2,GOLD
3,OIL


## 16. Data-quality checks

A researcher should be able to validate the data before trusting downstream results.

Typical checks include:

- duplicate intended keys;
- missing values;
- invalid ranges;
- unexpected category values;
- gaps in coverage;
- join mismatches.

These checks are often cheap to run in SQL before downloading the dataset.


In [24]:
# Duplicate intended keys
con.sql('''
SELECT date, asset, COUNT(*) AS n
FROM prices
GROUP BY date, asset
HAVING COUNT(*) > 1
ORDER BY n DESC
''').df()


,date,asset,n


In [25]:
# Coverage by asset
con.sql('''
SELECT
    asset,
    MIN(date) AS first_date,
    MAX(date) AS last_date,
    COUNT(*) AS n_obs,
    COUNT(close) AS n_non_null_close
FROM prices
GROUP BY asset
ORDER BY asset
''').df()


,asset,first_date,last_date,n_obs,n_non_null_close
0,BOND_EU,2024-01-02,2024-05-06,90,90
1,BOND_US,2024-01-02,2024-05-06,90,90
2,EQ_EU,2024-01-02,2024-05-06,90,90
3,EQ_US,2024-01-02,2024-05-06,90,90
4,EURUSD,2024-01-02,2024-05-06,90,90
5,GOLD,2024-01-02,2024-05-06,90,90
6,OIL,2024-01-02,2024-05-06,90,90
7,USDJPY,2024-01-02,2024-05-06,90,90


## 17. Query execution order: the mental model

A useful conceptual order is:

1. `FROM` / `JOIN`
2. `WHERE`
3. `GROUP BY`
4. aggregate calculations
5. `HAVING`
6. window functions
7. `SELECT`
8. `QUALIFY` where supported
9. `DISTINCT`
10. `ORDER BY`
11. `LIMIT`

The database optimizer may physically execute operations differently, but this logical model explains many syntax restrictions.

For example, a window result generally cannot be filtered directly in `WHERE` because `WHERE` conceptually happens earlier.


## 18. SQL versus pandas in a research workflow

SQL and pandas are complementary.

### Prefer SQL when

- the source data lives in a database or warehouse;
- filtering early can dramatically reduce transferred data;
- tables must be joined;
- the operation is a natural aggregation or window calculation;
- data quality should be checked close to the source;
- a transformation should be reproducible and shared across researchers.

### Prefer pandas / NumPy when

- the extracted dataset is already manageable in memory;
- the workflow is exploratory and changes rapidly;
- custom numerical logic is easier to express in Python;
- you are building signals, portfolio weights, backtests, statistical tests, or ML features;
- the operation depends on Python libraries outside the database.

A realistic workflow is therefore:

**database → SQL retrieval/transformation → pandas/NumPy research → sklearn/PyTorch when required**


## 19. Professional query style

Readable SQL matters because research queries are reviewed, reused, and debugged.

Recommended conventions:

- uppercase SQL keywords consistently;
- use descriptive table aliases (`p`, `s`, `a`) rather than arbitrary letters;
- qualify ambiguous columns;
- one selected expression per line in nontrivial queries;
- put each join condition on a separate logical line;
- use CTEs to name meaningful transformation stages;
- avoid `SELECT *` in durable production/research queries;
- document assumptions rather than obvious syntax;
- verify key uniqueness instead of using `DISTINCT` as a repair mechanism.

Example:


In [26]:
query = '''
WITH price_returns AS (
    SELECT
        p.date,
        p.asset,
        a.asset_class,
        p.close / LAG(p.close) OVER (
            PARTITION BY p.asset
            ORDER BY p.date
        ) - 1 AS ret
    FROM prices AS p
    INNER JOIN assets AS a
        ON p.asset = a.asset
),
class_summary AS (
    SELECT
        date,
        asset_class,
        AVG(ret) AS equal_weight_return,
        COUNT(ret) AS n_assets
    FROM price_returns
    GROUP BY date, asset_class
)
SELECT
    date,
    asset_class,
    equal_weight_return,
    n_assets
FROM class_summary
WHERE n_assets > 0
ORDER BY date, asset_class
'''

con.sql(query).df().head(12)


,date,asset_class,equal_weight_return,n_assets
0,2024-01-03,Commodity,-0.001468,2
1,2024-01-03,Equity,-0.002688,2
2,2024-01-03,FX,-0.005761,2
3,2024-01-03,Rates,-0.000994,2
4,2024-01-04,Commodity,0.001853,2
5,2024-01-04,Equity,-0.003615,2
6,2024-01-04,FX,-0.000018,2
7,2024-01-04,Rates,-0.000918,2
8,2024-01-05,Commodity,-0.001034,2
9,2024-01-05,Equity,-0.007475,2


## 20. Compact pandas ↔ SQL map

This table is only a conceptual bridge; the semantics are not always identical.

| pandas idea | SQL analogue |
|---|---|
| `df[cols]` | `SELECT cols` |
| boolean mask | `WHERE` |
| `sort_values` | `ORDER BY` |
| `drop_duplicates` | `DISTINCT` |
| `groupby(...).agg(...)` | `GROUP BY` + aggregates |
| `merge` | `JOIN` |
| `fillna` | `COALESCE` |
| `np.select` | `CASE` |
| grouped `shift(1)` | `LAG(...) OVER (...)` |
| grouped `shift(-1)` | `LEAD(...) OVER (...)` |
| `rank` | `RANK` / `DENSE_RANK` |
| quantile buckets | `NTILE` |
| grouped `rolling` | window aggregate with bounded `ROWS` |
| grouped `expanding` | window aggregate from `UNBOUNDED PRECEDING` |
| `concat` rows | `UNION ALL` |
| intermediate DataFrame | CTE (`WITH`) |
